In [0]:
from pyspark.sql.functions import *
from dateutil import parser

# ---------------- CONFIG ----------------
CATALOG = "chatbot_dev"
BRONZE_SCHEMA = "bronze"
SILVER_SCHEMA = "silver"

# Ensure silver schema exists
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SILVER_SCHEMA}")

df = spark.read.table(f"{CATALOG}.{BRONZE_SCHEMA}.patient_data")
clean_gender_format = df.withColumn(
    "Gender",
    when(col("Gender").isin("M", "Male"), "M").when(
        col("Gender").isin("F", "Female"), "F"
    ),
)

clean_date_format = clean_gender_format.withColumn(
    "dob",
    coalesce(
        try_to_date("dob", "MM-dd-yyyy"),
        try_to_date("dob", "dd-MM-yyyy"),
        try_to_date("dob", "yyyy-MM-dd"),
        try_to_date("dob", "yyyy/MM/dd"),   
        try_to_date("dob", "dd/MM/yyyy"),
        try_to_date("dob", "MM/dd/yyyy"),
        try_to_date("dob", "dd MMM, yyyy"),
        try_to_date("dob", "dd MMM yyyy"),   # <-- this parses 14 Nov 1975
        try_to_date("dob", "MMMM d, yyyy")
    )
).drop("_rescued_data")

# Removes duplicates
clean_dedup = clean_date_format.dropDuplicates(["patient_id"])

(clean_dedup.write
  .format("delta")
  .mode("overwrite")
  .option("overwriteSchema", "true")
  .saveAsTable(f"{CATALOG}.silver.patient_data")
)

In [0]:
%sql
select * from chatbot_dev.silver.patient_data